# 🚀 BuildFail-SLM: 01 - Data Loading & Profiling (Qwen2.5-Coder-7B-Instruct & 80k Dataset Pipeline)

This notebook handles:
1. **Hugging Face Hub Ingestion**: Downloading the primary code-diff benchmark (`ci-benchmark-user/ci-repair-bench`)
2. **Raw Dataset Inventory**: Verifying all 6 raw data sources in `data/01_raw/`
3. **80k Scale Preparation**: Inspecting raw sample capacities across GHALogs (1M+ runs) and Travis CI (2.6M+ builds)
4. **Schema & Type Profiling**: Inspecting real column types, structures, and missing values
5. **Deep-Dive Profiling**: Investigating real developer diffs, failure logs, and OOD benchmarks

In [6]:
import os
import shutil
from pathlib import Path
import pandas as pd
from huggingface_hub import hf_hub_download

RAW_DIR = Path("../data/01_raw") if Path("../data/01_raw").exists() else Path("data/01_raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# Step 1: Hugging Face Dataset Download (CI-REPAIR-BENCH)
# -------------------------------------------------------------------------
ci_repair_dir = RAW_DIR / "ci_repair_bench"
ci_repair_dir.mkdir(parents=True, exist_ok=True)
ci_repair_target = ci_repair_dir / "ci_repair_dataset.parquet"

hf_token = os.getenv("HUGGINGFACE_TOKEN") or None

if not ci_repair_target.exists() or ci_repair_target.stat().st_size == 0:
    print("Downloading ci_repair_dataset.parquet from Hugging Face Hub...")
    dataset_path = hf_hub_download(
        repo_id="ci-benchmark-user/ci-repair-bench",
        filename="ci_repair_dataset.parquet",
        repo_type="dataset",
        token=hf_token
    )
    shutil.copy(dataset_path, ci_repair_target)
    print(f"Successfully downloaded to: {ci_repair_target} ({ci_repair_target.stat().st_size / (1024*1024):.2f} MB)")
else:
    print(f"ci_repair_dataset.parquet already present: {ci_repair_target.stat().st_size / (1024*1024):.2f} MB")

ci_repair_dataset.parquet already present: 309.49 MB


## 2. Dataset Inventory Across All 6 Raw Sources

In [7]:
inventory = [
    {"Source": "CI-REPAIR-BENCH", "Path": RAW_DIR / "ci_repair_bench/ci_repair_dataset.parquet", "Format": "Parquet", "Role": "Primary Developer Code-Diff Supervision"},
    {"Source": "CI/CD Self-Healing (Main)", "Path": RAW_DIR / "cicd_self_healing/main/cicd_pipeline_failures_dataset.csv", "Format": "CSV", "Role": "Taxonomy, Root Causes & Fix Commands"},
    {"Source": "CI/CD Self-Healing (OOD)", "Path": RAW_DIR / "cicd_self_healing/ood/cicd_real_ood_world_dataset.csv", "Format": "CSV", "Role": "Isolated Out-Of-Distribution Benchmark"},
    {"Source": "LogChunks", "Path": RAW_DIR / "logchunks/LogChunks/LogChunks", "Format": "XML + Raw Logs", "Role": "Multilingual Failure Chunk Localization (28+ Langs)"},
    {"Source": "Java OSS Travis-CI", "Path": RAW_DIR / "java_travis/logcat-categories-and-patterns.tar.gz", "Format": "TAR.GZ + CSV", "Role": "Java Build & Compilation Error Patterns"},
    {"Source": "GHALogs", "Path": RAW_DIR / "ghalogs/runs.json.gz", "Format": "JSON.GZ", "Role": "GitHub Actions Workflow Runs & Errors"},
    {"Source": "Travis CI", "Path": RAW_DIR / "travis/19314170/final-2017-01-25.csv.gz", "Format": "CSV.GZ", "Role": "Large-scale Travis CI Failure Logs"}
]

for item in inventory:
    p = item["Path"]
    if p.exists():
        sz = p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.glob('**/*') if f.is_file())
        item["Size_MB"] = round(sz / (1024 * 1024), 2)
        item["Status"] = "Verified Available"
    else:
        item["Size_MB"] = 0.0
        item["Status"] = "Missing"

df_inventory = pd.DataFrame(inventory)
df_inventory

,Source,Path,Format,Role,Size_MB,Status
0,CI-REPAIR-BENCH,..\data\01_raw\ci_repair_bench\ci_repair_datas...,Parquet,Primary Developer Code-Diff Supervision,309.49,Verified Available
1,CI/CD Self-Healing (Main),..\data\01_raw\cicd_self_healing\main\cicd_pip...,CSV,"Taxonomy, Root Causes & Fix Commands",4.36,Verified Available
2,CI/CD Self-Healing (OOD),..\data\01_raw\cicd_self_healing\ood\cicd_real...,CSV,Isolated Out-Of-Distribution Benchmark,0.31,Verified Available
3,LogChunks,..\data\01_raw\logchunks\LogChunks\LogChunks,XML + Raw Logs,Multilingual Failure Chunk Localization (28+ L...,177.62,Verified Available
4,Java OSS Travis-CI,..\data\01_raw\java_travis\logcat-categories-a...,TAR.GZ + CSV,Java Build & Compilation Error Patterns,0.23,Verified Available
5,GHALogs,..\data\01_raw\ghalogs\runs.json.gz,JSON.GZ,GitHub Actions Workflow Runs & Errors,1014.13,Verified Available
6,Travis CI,..\data\01_raw\travis\19314170\final-2017-01-2...,CSV.GZ,Large-scale Travis CI Failure Logs,252.35,Verified Available


## 3. Profiling CI-REPAIR-BENCH
CI-REPAIR-BENCH provides developer-authored bug fixes (unified diffs) directly paired with CI failure logs and workflow names.

In [8]:
df_ci = pd.read_parquet(RAW_DIR / "ci_repair_bench/ci_repair_dataset.parquet")
print(f"Total CI-Repair records: {len(df_ci)}")
print("\nColumns:", list(df_ci.columns))
print("\nData Types:")
print(df_ci.dtypes)

sample_row = df_ci.iloc[0]
print(f"\nSample Record {sample_row.get('id')} ({sample_row.get('repo_owner')}/{sample_row.get('repo_name')}):")
print(f"Language: {sample_row.get('language')}, Error Type: {sample_row.get('error_type')}")
print("\n--- First 300 characters of real unified diff ---")
print(str(sample_row.get('diff', ''))[:300])

Total CI-Repair records: 565

Columns: ['language', 'id', 'repo_owner', 'repo_name', 'head_branch', 'workflow_name', 'workflow_filename', 'workflow_path', 'sha_fail', 'sha_success', 'workflow', 'logs', 'diff', 'changed_files', 'commit_link', 'error_type', 'commit_date', 'overall_jobs', 'failed_jobs', 'total_jobs', 'total_steps', 'total_failed_jobs', 'total_failed_steps', 'failure_types', 'failure_subtypes', 'num_failure_types', 'failure_details']

Data Types:
language                 str
id                       str
repo_owner               str
repo_name                str
head_branch              str
workflow_name            str
workflow_filename        str
workflow_path            str
sha_fail                 str
sha_success              str
workflow                 str
logs                  object
diff                     str
changed_files         object
commit_link              str
error_type            object
commit_date              str
overall_jobs          object
failed_jobs   

## 4. Profiling CI/CD Self-Healing Dataset (Main vs. OOD)
Contains structured failure taxonomy, root causes, and remediation/verification commands. The OOD set is reserved strictly for evaluation.

In [9]:
df_sh_main = pd.read_csv(RAW_DIR / "cicd_self_healing/main/cicd_pipeline_failures_dataset.csv")
df_sh_ood = pd.read_csv(RAW_DIR / "cicd_self_healing/ood/cicd_real_ood_world_dataset.csv")

print(f"Main Self-Healing Records: {len(df_sh_main)}")
print(f"OOD Evaluation Records:   {len(df_sh_ood)}")
print("\nMain Categories:")
print(df_sh_main["category"].value_counts().head(10))
print("\nSample Root Cause & Fix Command:")
print("Error Message:", df_sh_main.iloc[0]["error_message"][:150])
print("Root Cause:   ", df_sh_main.iloc[0]["root_cause"])
print("Fix Command:  ", df_sh_main.iloc[0]["fix_command"])

Main Self-Healing Records: 7777
OOD Evaluation Records:   489

Main Categories:
category
Network              765
Database             751
Security             710
Container Runtime    674
CI Orchestration     594
Runtime              575
Authorization        563
Storage              458
Infrastructure       450
Configuration        374
Name: count, dtype: int64

Sample Root Cause & Fix Command:
Error Message: [INFRA] time="2024-05-06T03:44:00Z" level=warning msg="x509: certificate is not yet valid; clock skew detected: delta=245s" host=ci-node-10 pid=46324
Root Cause:    NTP clock skew causing TLS certificate validation failures
Fix Command:   sudo systemctl stop ntp && sudo ntpdate -u pool.ntp.org && sudo systemctl start ntp


## 5. Profiling LogChunks, Java Travis & GHALogs
Multi-lingual log failure chunks, compilation errors, and CI runner failure traces.

In [10]:
import glob
import tarfile
import json

# 1. LogChunks
logchunks_dir = RAW_DIR / "logchunks/LogChunks/LogChunks/build-failure-reason"
xml_files = list(logchunks_dir.glob("*/*.xml"))
print(f"LogChunks XML build failure reason files: {len(xml_files)} across languages")

# 2. Java Travis
tar_path = RAW_DIR / "java_travis/logcat-categories-and-patterns.tar.gz"
if tar_path.exists():
    with tarfile.open(tar_path) as tar:
        json_members = [m for m in tar.getmembers() if m.name.endswith('.json')]
        print(f"Java Travis pattern files in tar.gz: {len(json_members)}")

# 3. GHALogs
import gzip
runs_path = RAW_DIR / "ghalogs/runs.json.gz"
if runs_path.exists():
    with gzip.open(runs_path, "rt", encoding="utf-8", errors="ignore") as f:
        sample_run = json.loads(f.readline())
        print(f"GHALogs sample run ID: {sample_run.get('id')}, Conclusion: {sample_run.get('conclusion')}")

LogChunks XML build failure reason files: 80 across languages
Java Travis pattern files in tar.gz: 12
GHALogs sample run ID: None, Conclusion: None
